# Snap2 Cursive OCR Fine-Tuning (PARSeq -> ONNX)

**Zero LLM tokens required.** Run this notebook on a free **Google Colab T4 GPU** (or Kaggle GPU) to fine-tune `baudm/parseq` on your 20,000-sample Snap2 cursive dataset (`training/dataset.zip`), track Validation CER each epoch, save `best_parseq.pt`, verify the exported ONNX graph with `onnxruntime`, and download a drop-in `parseq_cursive.onnx` for `Snap2/public/models/parseq/model.onnx`.

In [ ]:
# 1. Verify GPU and install dependencies
!nvidia-smi
!pip install -q timm pytorch-lightning onnx onnxruntime pillow kagglehub

In [ ]:
# 2. Clone official PARSeq repo
!git clone --depth 1 https://github.com/baudm/parseq.git
%cd parseq

In [ ]:
# 3. Load dataset.zip (from Google Drive recommended for ~80MB zip, or direct browser upload)
import zipfile
from pathlib import Path

USE_GOOGLE_DRIVE = False  # Set True if you uploaded dataset.zip to MyDrive/Snap2/dataset.zip
ZIP_PATH = '/content/drive/MyDrive/Snap2/dataset.zip'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    from google.colab import files
    uploaded = files.upload()
    ZIP_PATH = next(iter(uploaded.keys()))

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content/snap2_dataset')
print('Extracted dataset to /content/snap2_dataset')

In [ ]:
# 3B. Auto-download Kaggle cursive dataset (jessicamironyuk/cursivehandwriting) & stitch real-ink cursive words
import random, re, shutil, zipfile
from pathlib import Path
import numpy as np
from PIL import Image
import kagglehub

INCLUDE_KAGGLE_CURSIVE = True
STITCHED_WORD_COUNT = 4000
KAGGLE_OUT = Path('/content/kaggle_cursive')
KAGGLE_MANIFEST = KAGGLE_OUT / 'labels.tsv'

if INCLUDE_KAGGLE_CURSIVE:
    raw_dl = Path(kagglehub.dataset_download('jessicamironyuk/cursivehandwriting'))
    work_raw = Path('/content/kaggle_cursive_raw')
    if work_raw.exists():
        shutil.rmtree(work_raw)
    shutil.copytree(raw_dl, work_raw)

    # Unpack any nested zip archives (e.g. datafileszip.zip -> datafiles/)
    for zpath in list(work_raw.rglob('*.zip')):
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(zpath.parent / zpath.stem)

    # Prefer 'datafiles' folder per Kaggle dataset author's note
    df_dirs = [d for d in work_raw.rglob('datafiles') if d.is_dir()]
    scan_root = df_dirs[0] if df_dirs else work_raw

    def _infer_char_label(p: Path) -> str:
        parent = p.parent.name.strip()
        if len(parent) == 1 and parent.isalpha():
            return parent
        m_dir = re.match(r'^(?:upper|cap)[_-]?([a-zA-Z])$', parent, re.I)
        if m_dir:
            return m_dir.group(1).upper()
        m_dir_l = re.match(r'^(?:lower|small)[_-]?([a-zA-Z])$', parent, re.I)
        if m_dir_l:
            return m_dir_l.group(1).lower()
        stem = p.stem.strip()
        if len(stem) == 1 and stem.isalpha():
            return stem
        m_fn = re.match(r'^([a-zA-Z])(?:[_\-\d].*)$', stem)
        if m_fn:
            return m_fn.group(1)
        return ''

    by_char = {}
    rows = ['rel_path\tlabel\tsource\twriter_id']
    for ext in ('*.png', '*.jpg', '*.jpeg'):
        for img_p in sorted(scan_root.rglob(ext)):
            ch = _infer_char_label(img_p)
            if not ch:
                continue
            by_char.setdefault(ch.lower(), []).append(img_p)
            rows.append(f'{img_p.resolve().as_posix()}\t{ch}\tkaggle_cursive_char\tkaggle_real')

    print(f'Kaggle cursive chars found: {len(rows)-1} across {len(by_char)} letters ({sorted(by_char.keys())})')

    # Gather single-word vocabulary from Snap2 train.tsv to stitch real-ink cursive words
    vocab_words = set()
    snap2_train = Path('/content/snap2_dataset/splits/train.tsv')
    if snap2_train.is_file():
        for i, line in enumerate(snap2_train.read_text(encoding='utf-8').splitlines()):
            if i == 0:
                continue
            parts = line.split('\t')
            if len(parts) >= 2:
                w = parts[1].strip()
                if w.isalpha() and 3 <= len(w) <= 14 and all(c.lower() in by_char for c in w):
                    vocab_words.add(w)
    vocab_list = sorted(vocab_words)

    def _trim_glyph(pil_l: Image.Image, target_h: int) -> np.ndarray:
        arr = np.array(pil_l, dtype=np.uint8)
        if np.median(arr) < 127:
            arr = 255 - arr
        ys, xs = np.where(arr < 220)
        if len(xs) > 0:
            arr = arr[max(0, ys.min()-2):ys.max()+3, max(0, xs.min()-2):xs.max()+3]
        h, w = arr.shape[:2]
        new_w = max(8, int(round(w * (target_h / max(1, h)))))
        return np.array(Image.fromarray(arr).resize((new_w, target_h), Image.BICUBIC), dtype=np.uint8)

    stitched_dir = KAGGLE_OUT / 'stitched'
    stitched_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(42)
    stitched_n = 0
    if vocab_list and by_char:
        for idx in range(STITCHED_WORD_COUNT):
            word = rng.choice(vocab_list)
            th = rng.randint(28, 38)
            glyphs = []
            for ch in word:
                gp = rng.choice(by_char[ch.lower()])
                try:
                    glyphs.append(_trim_glyph(Image.open(gp).convert('L'), th))
                except Exception:
                    glyphs = []
                    break
            if not glyphs:
                continue
            pad = 10
            total_w = sum(g.shape[1] for g in glyphs) + pad * 2 + 4 * len(glyphs)
            canvas = np.full((th + pad * 2, total_w), 255, dtype=np.uint8)
            cx = pad
            for g in glyphs:
                gh, gw = g.shape
                cy = pad + rng.randint(-2, 2)
                cy = max(0, min(canvas.shape[0] - gh, cy))
                cx_end = min(canvas.shape[1], cx + gw)
                patch = canvas[cy:cy+gh, cx:cx_end]
                canvas[cy:cy+gh, cx:cx_end] = np.minimum(patch, g[:, :cx_end-cx])
                cx += gw + rng.randint(-2, 2)
            final_w = min(canvas.shape[1], max(cx + pad, 24))
            out_p = stitched_dir / f'kword_{idx:05d}.png'
            Image.fromarray(canvas[:, :final_w]).save(out_p)
            rows.append(f'{out_p.resolve().as_posix()}\t{word}\tkaggle_stitched_word\tkaggle_real')
            stitched_n += 1

    KAGGLE_OUT.mkdir(parents=True, exist_ok=True)
    KAGGLE_MANIFEST.write_text('\n'.join(rows) + '\n', encoding='utf-8')
    print(f'Saved {len(rows)-1} Kaggle samples ({stitched_n} stitched words) -> {KAGGLE_MANIFEST}')

# Optional: Auto-download IAM Handwriting Words (657 real writers, ~96k words) via kagglehub
INCLUDE_IAM_WORDS = True
IAM_MANIFEST = Path('/content/iam_words/labels.tsv')
if INCLUDE_IAM_WORDS:
    try:
        iam_root = Path(kagglehub.dataset_download('nibinv23/iam-handwriting-word-database'))
        words_txt = next(iam_root.rglob('words_new.txt'), None) or next(iam_root.rglob('words.txt'), None)
        iam_rows = ['rel_path\tlabel\tsource\twriter_id']
        if words_txt:
            img_index = {p.stem: p for p in iam_root.rglob('*.png')}
            for line in words_txt.read_text(encoding='utf-8', errors='ignore').splitlines():
                if not line or line.startswith('#'):
                    continue
                parts = line.split()
                if len(parts) >= 9 and parts[1] == 'ok':
                    wid, label = parts[0], parts[8].strip()
                    if wid in img_index and label and ' ' not in label:
                        iam_rows.append(f'{img_index[wid].resolve().as_posix()}\t{label}\tiam_real_word\tiam_{wid.split("-")[0]}')
            IAM_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
            IAM_MANIFEST.write_text('\n'.join(iam_rows) + '\n', encoding='utf-8')
            print(f'Loaded {len(iam_rows)-1} real IAM handwriting words -> {IAM_MANIFEST}')
    except Exception as e:
        print(f'Skipping IAM download ({e}) — continuing with Snap2 + Kaggle cursive datasets')

In [ ]:
# 4. Fine-tune pretrained PARSeq with per-epoch Validation CER & best-checkpoint saving
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True).to(device)
# Silence Lightning self.log() when running in a plain PyTorch loop (no Trainer attached)
model.log = lambda *args, **kwargs: None

img_transform = T.Compose([
    T.Resize((32, 128), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(0.5, 0.5),
])

def edit_distance(a: str, b: str) -> int:
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev if ca == cb else 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

class TsvOcrDataset(Dataset):
    def __init__(self, tsv_paths, valid_chars, max_len=25):
        if isinstance(tsv_paths, (str, Path)):
            tsv_paths = [tsv_paths]
        self.items = []
        skipped = 0
        for tsv_path in tsv_paths:
            p = Path(tsv_path)
            if not p.is_file():
                continue
            base = p.parent
            for i, line in enumerate(p.read_text(encoding='utf-8').splitlines()):
                if i == 0 and line.startswith('rel_path\t'):
                    continue
                parts = line.split('\t')
                if len(parts) >= 2:
                    label = parts[1].strip()
                    if not label or len(label) > max_len or not all(ch in valid_chars for ch in label):
                        skipped += 1
                        continue
                    raw_p = Path(parts[0].strip())
                    ip = raw_p if raw_p.is_absolute() else (base / raw_p).resolve()
                    src = parts[2].strip().lower() if len(parts) > 2 else 'synth'
                    is_real = any(k in src for k in ('real', 'kaggle', 'iam'))
                    if ip.is_file():
                        self.items.append((ip, label, is_real))
        if skipped:
            print(f'Filtered {skipped} multi-word/out-of-charset labels')
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        ip, label, _ = self.items[idx]
        img = Image.open(ip).convert('RGB')
        return img_transform(img), label

valid_chars = set(model.tokenizer._stoi.keys())
max_len = int(getattr(model.hparams, 'max_label_length', 25))
train_manifests = [
    '/content/snap2_dataset/splits/train.tsv',
    '/content/kaggle_cursive/labels.tsv',
    '/content/iam_words/labels.tsv',
]
train_ds = TsvOcrDataset(train_manifests, valid_chars, max_len)
val_ds = TsvOcrDataset('/content/snap2_dataset/splits/val.tsv', valid_chars, max_len)
n_real = sum(1 for _, _, r in train_ds.items if r)
n_synth = len(train_ds) - n_real
print(f'Loaded train={len(train_ds)} (real={n_real}, synth={n_synth}), val={len(val_ds)}')
assert len(train_ds) > 0 and len(val_ds) > 0, 'Dataset empty — check /content/snap2_dataset extraction!'

def _make_balanced_loader(ds, real_share: float, batch_size: int = 64, samples_per_epoch: int = 16000):
    r_cnt = max(1, sum(1 for _, _, r in ds.items if r))
    s_cnt = max(1, len(ds) - r_cnt)
    weights = [(real_share / r_cnt) if r else ((1.0 - real_share) / s_cnt) for _, _, r in ds.items]
    sampler = WeightedRandomSampler(weights, num_samples=min(samples_per_epoch, max(len(ds), 12000)), replacement=True)
    return DataLoader(ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)

stage1_loader = _make_balanced_loader(train_ds, real_share=0.50) if (n_real > 0 and n_synth > 0) else DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
stage2_loader = _make_balanced_loader(train_ds, real_share=0.80) if (n_real > 0 and n_synth > 0) else stage1_loader
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 10
best_cer = float('inf')
best_ckpt = '/content/best_parseq.pt'

for epoch in range(1, epochs + 1):
    # Stage 1 (Epochs 1-7): 50/50 Real+Synth @ 1e-4 | Stage 2 (Epochs 8-10): 80/20 Real-Ink Polish @ 2e-5
    is_stage2 = epoch >= 8
    active_loader = stage2_loader if is_stage2 else stage1_loader
    target_lr = 2e-5 if is_stage2 else 1e-4
    for pg in optimizer.param_groups:
        pg['lr'] = target_lr

    model.train()
    total_loss = 0.0
    for imgs, labels in active_loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.training_step((imgs, list(labels)), 0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 20.0)
        optimizer.step()
        total_loss += float(loss.item())

    # Validation CER check
    model.eval()
    tot_edits, tot_chars = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            logits = model(imgs.to(device))
            preds, _ = model.tokenizer.decode(logits.softmax(-1))
            for pred, gt in zip(preds, labels):
                tot_edits += edit_distance(pred.lower(), gt.lower())
                tot_chars += max(1, len(gt))
    val_cer = tot_edits / max(1, tot_chars)
    stage_tag = 'S2(80/20)' if is_stage2 else 'S1(50/50)'
    print(f'Epoch {epoch:02d}/{epochs} [{stage_tag}] | Loss: {total_loss / max(1, len(active_loader)):.4f} | Val CER: {val_cer*100:.2f}%')
    if val_cer < best_cer:
        best_cer = val_cer
        torch.save(model.state_dict(), best_ckpt)
        print(f'  -> Saved new best checkpoint ({best_cer*100:.2f}% CER)')

In [ ]:
# 5. Export best checkpoint to ONNX, verify with onnxruntime, and download
import onnxruntime as ort
import numpy as np
from google.colab import files

model.load_state_dict(torch.load(best_ckpt, map_location='cpu'))
model.eval().cpu()
# Disable autoregressive loop (which emits unsupported bool Where nodes in ONNX CPU)
model.model.decode_ar = False
dummy = torch.randn(1, 3, 32, 128)
out_onnx = '/content/parseq_cursive.onnx'

for ref_iters in (1, 0):
    model.model.refine_iters = ref_iters
    torch.onnx.export(
        model,
        dummy,
        out_onnx,
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
        opset_version=14,
        do_constant_folding=True,
        dynamo=False,
    )
    try:
        sess = ort.InferenceSession(out_onnx, providers=['CPUExecutionProvider'])
        print(f'ONNX export verified on CPU with decode_ar=False, refine_iters={ref_iters}')
        break
    except Exception as e:
        print(f'refine_iters={ref_iters} failed ORT check ({e}), trying next...')
else:
    raise RuntimeError('ONNX export failed ORT CPU verification')

# Save matching tokenizer charset config for Snap2
out_cfg = '/content/parseq_cursive_config.json'
with open(out_cfg, 'w', encoding='utf-8') as f:
    json.dump({'itos': list(model.tokenizer._itos), 'img_size': [32, 128]}, f, indent=2)

# Sanity-check exported ONNX graph on 5 validation samples
in_name = sess.get_inputs()[0].name
for i in range(min(5, len(val_ds))):
    x, gt = val_ds[i]
    logits = sess.run(None, {in_name: x.unsqueeze(0).numpy()})[0]
    pred, _ = model.tokenizer.decode(torch.from_numpy(logits).softmax(-1))
    print(f'ONNX check [{i}]: gt={gt!r} -> pred={pred[0]!r}')

files.download(out_onnx)